# HypatiaX Notebook 02: Pure LLM Baseline Experiments

**Paper:** LLMs as Interfaces to Symbolic Discovery: Perfect Extrapolation via Hybrid Architectures  
**Journal:** Journal of Machine Learning Research (2026)  
**Author:** Dr. Ruperto Pedro Bonet Chaple

This notebook documents the Pure LLM baseline results — cases where the LLM directly proposes the formula without symbolic regression.

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# ── Repo-root resolution ─────────────────────────────────────────────────────
# Works whether the notebook is run from the repo root, the notebooks/ dir,
# or any subdirectory.  Looks for the marker file hypatiax/data/results.
def _find_repo_root() -> Path:
    """Walk up from this notebook until we find hypatiax/data/results."""
    here = Path().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'results').exists():
            return candidate
        if (candidate / 'hypatiax' / 'data' / 'results').exists():
            return candidate / 'hypatiax'
    raise FileNotFoundError(
        "Cannot locate repo root.  "
        "Run the notebook from inside the LLM-HypatiaX-PAPERS repository."
    )

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'data' / 'results'
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so hypatiax modules are importable
if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

print(f"✓ Repo root   : {REPO_ROOT}")
print(f"✓ Results dir : {RESULTS_DIR}")
print(f"✓ Figures dir : {FIGURES_DIR}")

MAIN_FILE    = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_131438.json'
MAIN_FILE2   = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_133510.json'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-paper')
sns.set_palette('husl')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})


with open(MAIN_FILE) as f:
    raw_results = json.load(f)

df = pd.DataFrame(raw_results)

# Extract fields
df['equation_name'] = df['metadata'].apply(lambda x: x.get('equation_name', ''))
df['difficulty']    = df['metadata'].apply(lambda x: x.get('difficulty', ''))
df['formula_type']  = df['metadata'].apply(lambda x: x.get('formula_type', ''))
df['ground_truth']  = df['metadata'].apply(lambda x: x.get('ground_truth', ''))
df['llm_r2']        = df['llm_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['llm_rmse']      = df['llm_result'].apply(lambda x: x['metrics']['rmse'] if x else None)
df['llm_success']   = df['llm_result'].apply(lambda x: x['metrics'].get('success', False) if x else False)
df['llm_formula']   = df['llm_result'].apply(lambda x: x.get('formula', '') if x else '')

print(f'✓ Loaded {len(df)} equations')

## 2. LLM Performance Overview

In [ ]:
# Filter valid LLM R² (exclude extreme outliers from failed runs)
llm_valid = df[df['llm_r2'].notna() & (df['llm_r2'] >= -10)].copy()
llm_excellent = df[df['llm_r2'] >= 0.95]
llm_failed    = df[df['llm_r2'] < 0.95]

print('=== PURE LLM BASELINE RESULTS ===')
print(f'Total equations:          {len(df)}')
print(f'LLM excellent (R²≥0.95):  {len(llm_excellent)} ({len(llm_excellent)/len(df)*100:.1f}%)')
print(f'LLM struggled (R²<0.95):  {len(llm_failed)} ({len(llm_failed)/len(df)*100:.1f}%)')
print(f'\nMean LLM R² (valid):      {llm_valid["llm_r2"].mean():.4f}')
print(f'Median LLM R² (valid):    {llm_valid["llm_r2"].median():.4f}')

## 3. LLM Results by Domain

In [ ]:
domain_llm = llm_valid.groupby('domain').agg(
    count=('llm_r2', 'count'),
    mean_r2=('llm_r2', 'mean'),
    excellent=('llm_r2', lambda x: (x >= 0.95).sum())
).round(4)
domain_llm['success_rate'] = (domain_llm['excellent'] / domain_llm['count'] * 100).round(1)

print('LLM Performance by Domain:')
display(domain_llm)

## 4. LLM Results by Difficulty

In [ ]:
diff_llm = llm_valid.groupby('difficulty').agg(
    count=('llm_r2', 'count'),
    mean_r2=('llm_r2', 'mean'),
    excellent=('llm_r2', lambda x: (x >= 0.95).sum())
).round(4)
diff_llm['success_rate'] = (diff_llm['excellent'] / diff_llm['count'] * 100).round(1)

print('LLM Performance by Difficulty:')
display(diff_llm)

## 5. Cases Where LLM Excelled vs Struggled

In [ ]:
print('=== LLM EXCELLENT (R² ≥ 0.95) ===')
cols = ['equation_name', 'domain', 'difficulty', 'llm_r2', 'llm_formula', 'ground_truth']
display(llm_excellent[cols].sort_values('llm_r2', ascending=False).reset_index(drop=True))

print('\n=== LLM STRUGGLED (R² < 0.95) ===')
display(llm_failed[cols].sort_values('llm_r2', ascending=False).reset_index(drop=True))

## 6. Visualize LLM Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) R² distribution
ax = axes[0]
valid_r2 = llm_valid['llm_r2'].clip(-1, 1)
ax.hist(valid_r2, bins=20, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(x=0.95, color='red', linestyle='--', linewidth=2, label='Threshold (0.95)')
ax.set_xlabel('LLM R² Score')
ax.set_ylabel('Frequency')
ax.set_title('(a) LLM R² Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Success rate by domain
ax = axes[1]
domain_llm['success_rate'].plot(kind='bar', ax=ax, color='#2ecc71', alpha=0.85)
ax.set_title('(b) LLM Success Rate by Domain')
ax.set_ylabel('Success Rate (%)')
ax.set_xlabel('Domain')
ax.tick_params(axis='x', rotation=45)
ax.set_ylim([0, 110])
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

# (c) Success rate by difficulty
ax = axes[2]
colors_diff = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c'}
bars = ax.bar(diff_llm.index, diff_llm['success_rate'],
              color=[colors_diff.get(d, '#3498db') for d in diff_llm.index], alpha=0.85)
ax.set_title('(c) LLM Success Rate by Difficulty')
ax.set_ylabel('Success Rate (%)')
ax.set_xlabel('Difficulty')
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.3, axis='y')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.suptitle('Pure LLM Baseline Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_pure_llm_performance.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / '02_pure_llm_performance.png', dpi=300, bbox_inches='tight')
print('✓ Saved LLM performance figure')
plt.show()

## 7. Why LLM Struggles on Some Equations

The LLM struggles primarily when:
- Physical constants have scale mismatches (e.g., Coulomb's constant ~9×10⁹)
- Equations involve non-normalized units
- Multi-term additive structures (e.g., Bernoulli's equation)

In these cases, the Hybrid system falls back to Neural Network or Symbolic Regression.

In [ ]:
# Show decision routing
print('Decision routing (when LLM struggled):')
print(df[df['decision'] == 'nn'][['equation_name', 'domain', 'difficulty', 'decision_reason']]
      .reset_index(drop=True).to_string())

---
**Previous:** [01_data_generation.ipynb](01_data_generation.ipynb)  
**Next:** [03_hybrid_experiments.ipynb](03_hybrid_experiments.ipynb)